# Block 2 — One measurement: size, resolution, and sensitivity
### How big is a single radar value, and can the radar even detect it?

Block 1 showed *where* the beam goes. Now: a measurement is a chunk of air with a real size, set partly by the beam (Block 1) and partly by the pulse — and whether you can detect it at all comes from the radar equation. Sliders only; run setup first.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

C = 2.998e8
kB, T0 = 1.380649e-23, 290.0

def beamwidth(freq_ghz=2.8, dish_m=8.5):
    return np.degrees(1.27 * (C/(freq_ghz*1e9)) / dish_m)

def min_Z(r_km, power_kW=750, dish_m=8.5, freq_ghz=2.8, pulse_us=1.57,
          eta=0.45, NF_dB=4, Kw2=0.93, snr_dB=3):
    # minimum detectable reflectivity (dBZ) vs range, from the weather radar equation
    lam = C/(freq_ghz*1e9); tau = pulse_us*1e-6; r = np.asarray(r_km, float)*1e3
    G = eta*(np.pi*dish_m/lam)**2; theta = 1.27*lam/dish_m
    N = kB*T0*(1/tau)*10**(NF_dB/10); snr = 10**(snr_dB/10)
    K = (np.pi**3 * C * power_kW*1e3 * G**2 * theta**2 * tau * Kw2)/(1024*np.log(2)*lam**2)
    return 10*np.log10((snr*N*r**2/K)/1e-18)

print("Block 2 ready | range depth at 1.57 us:", round(C*1.57e-6/2), "m | MDS @100 km:", round(float(min_Z(100)),1), "dBZ")

Block 2 ready | range depth at 1.57 us: 235 m | MDS @100 km: -2.9 dBZ


## 1 — The pulse: one slider, three consequences

Pulse length τ pulls three ways at once: **range resolution** Δr = cτ/2, **sensitivity** (∝ τ — a longer pulse carries more energy), and the **blind range** near the radar (it can't listen while transmitting). Two targets sit in front of the radar — lengthen the pulse and watch them blur together.

In [2]:
def plot_pulse(pulse_us=1.57, separation_m=400):
    dr_m = C*pulse_us*1e-6/2; r = np.linspace(0, 4, 2000)
    sigma = (dr_m/1000)/2.355; t1, t2 = 2.0, 2.0 + separation_m/1000
    profile = np.exp(-0.5*((r-t1)/sigma)**2) + np.exp(-0.5*((r-t2)/sigma)**2)
    plt.figure(figsize=(8, 4.6))
    plt.axvspan(0, dr_m/1000, color="gray", alpha=0.2, label="blind range")
    plt.plot(r, profile, lw=2)
    plt.axvline(t1, color="k", ls=":", lw=0.8); plt.axvline(t2, color="k", ls=":", lw=0.8)
    plt.xlim(0, 4); plt.ylim(0, 2.1); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range (km)"); plt.ylabel("relative power")
    plt.title("dr = " + str(round(dr_m)) + " m  |  blind range " + str(round(dr_m)) + " m  |  sensitivity "
              + format(10*np.log10(pulse_us/1.57), "+.1f") + " dB vs short pulse")
    plt.show()

interact(plot_pulse,
         pulse_us=FloatSlider(value=1.57, min=0.5, max=5, step=0.1, description="pulse µs"),
         separation_m=FloatSlider(value=400, min=50, max=1500, step=50, description="sep m"));

interactive(children=(FloatSlider(value=1.57, description='pulse µs', max=5.0, min=0.5), FloatSlider(value=400…

**Basic**

1. With the targets 400 m apart, are they two peaks at the short pulse (1.57 µs)? Lengthen τ until they merge — what is Δr there, versus 400 m?
2. Read the blind range at 1.57 µs and at 4.7 µs. Which pulse sees closer to the radar?

**A little further**

1. By how many dB does sensitivity change from 1.57 to 4.7 µs? Which pulse would you use to hunt weak clear-air echo — and what do you give up?
2. Two targets just separate when their spacing ≈ Δr. Test this: find the merge separation at two different pulse lengths and compare to cτ/2.

## 2 — The whole sample: the resolution volume

Combine the beam (Block 1) with the pulse: one measurement is a box ≈ r·θ wide **and** tall (grows with range) by Δr = cτ/2 deep (fixed by the pulse). Far away, that's a surprisingly large chunk of air to average into a single number.

In [3]:
def plot_volume(range_km=120, pulse_us=1.57):
    bw = beamwidth()
    width = range_km*np.radians(bw); depth = C*pulse_us*1e-6/2/1000
    volume = depth*np.pi*(width/2)**2; r = np.linspace(1, 230, 300)
    plt.figure(figsize=(8, 4.5))
    plt.plot(r, r*np.radians(bw)*1000, lw=2, label="across-beam = vertical (r·θ)")
    plt.axhline(C*pulse_us*1e-6/2, color="tab:orange", lw=2, label="range depth (cτ/2)")
    plt.axvline(range_km, color="tab:red", ls=":")
    plt.ylim(0, 4000); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range (km)"); plt.ylabel("dimension (m)")
    plt.title("at " + str(range_km) + " km: " + str(round(width,1)) + " km wide/tall x "
              + str(round(depth*1000)) + " m deep ~ " + str(round(volume,2)) + " km³")
    plt.show()

interact(plot_volume,
         range_km=IntSlider(value=120, min=10, max=230, step=10, description="range km"),
         pulse_us=FloatSlider(value=1.57, min=0.5, max=5, step=0.1, description="pulse µs"));

interactive(children=(IntSlider(value=120, description='range km', max=230, min=10, step=10), FloatSlider(valu…

**Basic**

1. How wide and tall is the sample at 60 km versus 230 km?
2. The radial depth stays ~250 m at every range. Which slider changes it?

**A little further**

1. A convective core is ~2 km across. Beyond what range does the beam grow wider than the storm, so the peak reads too low (beam filling)?
2. Roughly how many km³ does one pixel average at 150 km versus 230 km? Why does that matter for rainfall estimates and echo tops?

## 3 — Can you even detect it? The radar equation

Received power scales as P_t·D²·τ·Z / (λ⁴ r²), so the faintest echo the radar can detect — the minimum detectable Z — climbs with range. Slide the hardware and watch the whole sensitivity curve move; drop a target on and see whether it clears the noise.

In [5]:
def plot_radar_eq(power_kW=750, dish_m=8.5, freq_ghz=2.8, pulse_us=1.57, target_dBZ=10, target_km=150):
    r = np.linspace(10, 460, 400); zmin = min_Z(r, power_kW, dish_m, freq_ghz, pulse_us)
    bw = np.degrees(1.27*(C/(freq_ghz*1e9))/dish_m)
    detect = target_dBZ >= np.interp(target_km, r, zmin)
    plt.figure(figsize=(8, 4.6))
    plt.plot(r, zmin, "k", lw=2, label="min detectable Z")
    plt.fill_between(r, zmin, -30, alpha=0.05)
    for z, lab in [(10, "drizzle"), (20, "light rain"), (45, "heavy rain")]:
        plt.axhline(z, color="gray", ls=":", lw=0.6); plt.text(455, z+0.6, lab, fontsize=7, ha="right", color="gray")
    plt.plot([target_km], [target_dBZ], "o", ms=11, color="tab:green" if detect else "tab:red")
    plt.annotate("detected" if detect else "below noise", (target_km, target_dBZ), (target_km+8, target_dBZ+5), fontsize=9)
    plt.ylim(-30, 55); plt.grid(alpha=0.3); plt.legend(loc="lower right")
    plt.xlabel("range (km)"); plt.ylabel("reflectivity (dBZ)")
    plt.title("beamwidth " + str(round(bw,2)) + "°  |  MDS @100 km = "
              + str(round(float(min_Z(100, power_kW, dish_m, freq_ghz, pulse_us)),1)) + " dBZ")
    plt.show()

interact(plot_radar_eq,
         power_kW=FloatSlider(value=750, min=100, max=1500, step=50, description="power kW"),
         dish_m=FloatSlider(value=8.5, min=2, max=15, step=0.5, description="dish m"),
         freq_ghz=FloatSlider(value=2.8, min=2, max=10, step=0.2, description="freq GHz"),
         pulse_us=FloatSlider(value=1.57, min=0.5, max=5, step=0.1, description="pulse µs"),
         target_dBZ=FloatSlider(value=10, min=-20, max=50, step=5, description="target dBZ"),
         target_km=FloatSlider(value=150, min=20, max=450, step=10, description="target km"));

interactive(children=(FloatSlider(value=750.0, description='power kW', max=1500.0, min=100.0, step=50.0), Floa…

**Basic**

1. Read the minimum detectable Z at 100 km. Now double the transmit power — how many dBZ does it improve?
2. Put a +10 dBZ echo (light rain) at 450 km. Detected? Which single slider rescues it most easily?

**A little further**

1. Keep the dish fixed and move S-band → X-band (2.8 → 9.4). How many dB does the sensitivity change, and how does that match the 1/λ⁴ scaling?
2. Find a *smaller dish + shorter wavelength* combo that matches NEXRAD's 100 km sensitivity. So why can a compact X-band radar be sensitive — and what's the catch you'll meet in Block 4?

## Facilitator notes
- This block ties the beam (Block 1) and the pulse together into a single sample, then asks whether it's detectable — the natural bridge from geometry to signals.
- The radar equation widget replaces a separate 'sensitivity vs range' plot — the minimum-detectable-Z curve *is* the radar equation.
- The one idea to leave with: *power is the weak knob; aperture and wavelength dominate sensitivity — and a 'measurement' is a big spatial average, not a point.*